# 04 - Model 2: frozen sentence-transformer embeddings + classifier

`all-MiniLM-L6-v2` encodes each posting into a 384-dim vector. The encoder stays
**frozen** - no fine-tuning - and a plain classifier is trained on top.

This is the cheap middle ground between bag-of-words and a fine-tuned transformer:
it understands wording that TF-IDF cannot, but trains in seconds and runs on CPU.

**Cost warning.** Encoding ~17.9k postings takes roughly 5-15 minutes on CPU and
under a minute on a GPU. The embeddings are cached to `models/`, so you pay that
once. Set `DEVICE = "cuda"` below if you have a GPU.

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether this runs from notebooks/ or the project root.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, evaluation, features, models, preprocessing

config.set_seed()
config.ensure_dirs()
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

In [ ]:
DEVICE = None  # None = auto (CPU unless torch finds a GPU); or "cuda" / "cpu"

train, val, test = preprocessing.load_splits()
y_train, y_val, y_test = (part[config.TARGET].to_numpy() for part in (train, val, test))
print(f"encoder: {config.SENTENCE_TRANSFORMER_MODEL}")

### Encode once, cache to disk

In [ ]:
def encode_cached(split_name, texts):
    """Encode a split, reusing models/embeddings_<split>.npy if it already exists."""
    path = config.MODELS_DIR / f"embeddings_{split_name}.npy"
    if path.exists():
        cached = np.load(path)
        print(f"{split_name}: loaded cached {cached.shape} from {path.name}")
        return cached
    print(f"{split_name}: encoding {len(texts):,} postings (this is the slow part)...")
    embeddings = models.encode_texts(texts, device=DEVICE)
    np.save(path, embeddings)
    print(f"{split_name}: saved {embeddings.shape} -> {path.name}")
    return embeddings


emb_train = encode_cached("train", train[config.FULL_TEXT_COLUMN].tolist())
emb_val = encode_cached("val", val[config.FULL_TEXT_COLUMN].tolist())
emb_test = encode_cached("test", test[config.FULL_TEXT_COLUMN].tolist())

In [ ]:
# Name the columns and use the same frame for fit and predict - keeps sklearn and
# LightGBM from complaining about anonymous features.
X_train = models.embeddings_to_frame(emb_train)
X_val = models.embeddings_to_frame(emb_val)
X_test = models.embeddings_to_frame(emb_test)
print(f"X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")

## Classifier A: logistic regression on the embeddings

In [ ]:
logreg = models.train_embedding_classifier(X_train, y_train, kind="logreg")

proba_train = models.predict_proba(logreg, X_train)
proba_val = models.predict_proba(logreg, X_val)

train_metrics = evaluation.evaluate_predictions(y_train, proba_train)
val_metrics = evaluation.evaluate_predictions(y_val, proba_val)
print("train:", {k: round(v, 4) for k, v in train_metrics.items()})
print("val:  ", {k: round(v, 4) for k, v in val_metrics.items()})

In [ ]:
best_threshold, best_f1 = evaluation.tune_threshold(y_val, proba_val, beta=1.0)
print(f"best F1 threshold: {best_threshold:.3f} (F1={best_f1:.3f})")

val_tuned = evaluation.evaluate_predictions(y_val, proba_val, threshold=best_threshold)
evaluation.log_experiment("minilm_frozen_logreg", val_tuned, split="val",
                          notes="all-MiniLM-L6-v2 frozen, LogReg head, F1-tuned")

proba_test = models.predict_proba(logreg, X_test)
test_metrics = evaluation.evaluate_predictions(y_test, proba_test, threshold=best_threshold)
print("test: ", {k: round(v, 4) for k, v in test_metrics.items()})

evaluation.log_experiment("minilm_frozen_logreg", test_metrics, split="test",
                          notes="threshold frozen from val")
models.save_test_predictions("minilm_frozen_logreg", y_test, proba_test)
models.save_model(logreg, "minilm_frozen_logreg")

## Classifier B: LightGBM on the same embeddings

Same features, non-linear head. Worth one cell to see whether the extra capacity
buys anything on 384 dense dimensions.

In [ ]:
lgbm_head = models.train_embedding_classifier(X_train, y_train, kind="lightgbm")
proba_val_lgbm = models.predict_proba(lgbm_head, X_val)

lgbm_val_metrics = evaluation.evaluate_predictions(y_val, proba_val_lgbm)
lgbm_threshold, _ = evaluation.tune_threshold(y_val, proba_val_lgbm, beta=1.0)
lgbm_val_tuned = evaluation.evaluate_predictions(y_val, proba_val_lgbm, threshold=lgbm_threshold)

print("LogReg head  val AP:", round(val_metrics["average_precision"], 4))
print("LightGBM head val AP:", round(lgbm_val_metrics["average_precision"], 4))

evaluation.log_experiment("minilm_frozen_lightgbm", lgbm_val_tuned, split="val",
                          notes="all-MiniLM-L6-v2 frozen, LightGBM head, F1-tuned")

proba_test_lgbm = models.predict_proba(lgbm_head, X_test)
lgbm_test_metrics = evaluation.evaluate_predictions(y_test, proba_test_lgbm,
                                                    threshold=lgbm_threshold)
evaluation.log_experiment("minilm_frozen_lightgbm", lgbm_test_metrics, split="test",
                          notes="threshold frozen from val")
models.save_test_predictions("minilm_frozen_lightgbm", y_test, proba_test_lgbm)

In [ ]:
ax = evaluation.plot_pr_curve(y_test, proba_test, label="MiniLM + LogReg")
evaluation.plot_pr_curve(y_test, proba_test_lgbm, label="MiniLM + LightGBM", ax=ax,
                         save_as="15_minilm_pr_curve.png")
plt.show()

evaluation.plot_confusion_matrix(y_test, (proba_test >= best_threshold).astype(int),
                                 title=f"MiniLM + LogReg (test, t={best_threshold:.2f})",
                                 save_as="16_minilm_confusion_matrix.png")
plt.show()

## Compare every model scored so far

Each notebook wrote `models/predictions_<name>_test.csv` on the **same test split**,
so the curves below are directly comparable. Whatever has been run shows up; the
DistilBERT row appears once you drop its predictions file in from Colab.

In [ ]:
prediction_files = sorted(config.MODELS_DIR.glob("predictions_*_test.csv"))
if not prediction_files:
    print("No prediction files yet - run notebooks 02 and 03 first.")
else:
    ax = None
    summary = []
    for path in prediction_files:
        name = path.stem.replace("predictions_", "").replace("_test", "")
        frame = pd.read_csv(path)
        ax = evaluation.plot_pr_curve(frame["y_true"], frame["y_proba"], label=name, ax=ax)
        metrics = evaluation.evaluate_predictions(frame["y_true"], frame["y_proba"])
        summary.append({"model": name,
                        "average_precision": metrics["average_precision"],
                        "roc_auc": metrics["roc_auc"]})
    ax.set_title("Test-set PR curves - all models, identical rows")
    ax.figure.savefig(config.FIGURES_DIR / "17_model_comparison_pr.png", dpi=150,
                      bbox_inches="tight")
    plt.show()
    print(pd.DataFrame(summary).sort_values("average_precision", ascending=False)
          .round(4).to_string(index=False))

In [ ]:
evaluation.load_experiments()

Next: `05_distilbert_colab.ipynb` - upload it to Colab with a GPU runtime, along
with `data/processed/*.csv`.